In [1]:
import re
import numpy as np
import tensorflow as tf
import os # Import os module to handle paths

# Install konlpy and its dependencies
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install --upgrade pip

# Force uninstall to ensure a clean slate before reinstallation
!pip uninstall -y konlpy JPype1 mecab-python3 > /dev/null

!pip install 'setuptools<60.0.0' # Crucial for some legacy build processes or dependencies

# Install system MeCab and Korean dictionary using the konlpy provided script
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

# Install JPype1 and mecab-python3 first, then konlpy, as konlpy relies on these.
!pip install JPype1
!pip install mecab-python3 # This package provides the 'MeCab.Tagger' that konlpy expects.
!pip install konlpy
# !pip install gensim # Moved to a separate cell (5a1aad9f) for better isolation

# The from konlpy.tag import Mecab will now be executed after all dependencies are installed
from konlpy.tag import Mecab

# ==========================================
# Step 2. 데이터 정제 (Data Cleaning)
# ==========================================

def preprocessing(sent, lang="kor"):
    sent = sent.lower().strip()
    if lang == "kor":
        # 한글, 영문, 숫자의 주요 문장부호 정규식 처리
        sent = re.sub(r"[^?.!,a-z0-9가-힣ㄱ-ㅎㅏ-ㅣ\s]", "", sent)
        sent = re.sub(r"([?.!,])", r" \1 ", sent)
        sent = re.sub(r'[" "]+', " ", sent)
    else:
        sent = re.sub(r"[^?.!,a-z0-9\s]", "", sent)
        sent = re.sub(r"([?.!,])", r" \1 ", sent)
        sent = re.sub(r'[" "]+', " ", sent)
    return sent.strip()

def clean_and_filter_corpus(kor_list, eng_list):
    # set 자료형을 활용하여 중복 병렬 쌍 제거
    cleaned_set = set(zip(kor_list, eng_list))

    # Explicitly set the MeCab dictionary path
    # The default installation path for mecab-ko-dic by konlpy's script is usually /usr/local/lib/mecab/dic/mecab-ko-dic
    mecab_dic_path = os.path.join(os.sep, 'usr', 'local', 'lib', 'mecab', 'dic', 'mecab-ko-dic')

    # Initialize Mecab. If the dictionary path is valid, use it. Otherwise, try without it.
    # konlpy's Mecab class should now be able to find Tagger from the installed mecab-python3.
    if os.path.exists(mecab_dic_path):
        mecab = Mecab(dicpath=mecab_dic_path)
    else:
        print(f"Warning: MeCab dictionary not found at {mecab_dic_path}. Attempting to initialize Mecab without explicit dicpath.")
        mecab = Mecab()

    kor_corpus = []
    eng_corpus = []

    for kor, eng in cleaned_set:
        kor_clean = preprocessing(kor, "kor")
        eng_clean = preprocessing(eng, "eng")

        # 영문 문장엔 <start>와 <end> 토큰 추가
        eng_tokenized = f"<start> {eng_clean} <end>".split()
        # 한글 토큰화 (KoNLPy Mecab 사용)
        kor_tokenized = mecab.morphs(kor_clean)

        # 토큰 길이 40 이하인 데이터만 선별
        if len(kor_tokenized) <= 40 and len(eng_tokenized) <= 40:
            if len(kor_tokenized) > 0 and len(eng_tokenized) > 2:
                kor_corpus.append(kor_tokenized)
                eng_corpus.append(eng_tokenized)

    return kor_corpus, eng_corpus

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
mecab-ko is al

### Step 2.1. 데이터셋 로드 (Load Dataset)

먼저 실제 뉴스 한글-영문 병렬 코퍼스를 업로드해야 합니다. 아래 셀의 `kor_list`와 `eng_list` 변수를 실제 데이터셋으로 대체해주세요. 파일이 Colab 환경에 없다면, 직접 업로드하거나 Google Drive에서 로드할 수 있습니다.

예를 들어, 두 개의 텍스트 파일(하나의 문장 당 한 줄)로 되어 있는 경우:

1.  **파일 업로드**: Colab 왼쪽 파일 아이콘을 클릭하여 파일을 직접 업로드하거나, Google Drive를 마운트하여 파일을 불러올 수 있습니다.
2.  **데이터 로드**: `kor_list = open('korean.txt', encoding='utf-8').read().splitlines()` 와 같이 파일을 읽어 리스트로 변환합니다.

In [2]:
# Ensure gensim is installed as it's a dependency for loading FastText embeddings
!pip install gensim

In [3]:
import os
import re
import shutil # For potential cleanup if git clone fails partially
from konlpy.tag import Mecab
import gensim.downloader as api # For downloading pre-trained embeddings
import numpy as np

# ==========================================
# Step 2. 데이터 정제 (Data Cleaning)
# These functions are moved here to ensure they are defined before use.
# ==========================================

def preprocessing(sent, lang="kor"):
    sent = sent.lower().strip()
    if lang == "kor":
        # 한글, 영문, 숫자의 주요 문장부호 정규식 처리
        sent = re.sub(r"[^?.!,a-z0-9가-힣ㄱ-ㅎㅏ-ㅣ\s]", "", sent)
        sent = re.sub(r"([?.!,])", r" \1 ", sent)
        sent = re.sub(r'[" "]+', " ", sent)
    else:
        sent = re.sub(r"[^?.!,a-z0-9\s]", "", sent)
        sent = re.sub(r"([?.!,])", r" \1 ", sent)
        sent = re.sub(r'[" "]+', " ", sent)
    return sent.strip()

def clean_and_filter_corpus(kor_list, eng_list):
    # set 자료형을 활용하여 중복 병렬 쌍 제거
    cleaned_set = set(zip(kor_list, eng_list))

    # Explicitly set the MeCab dictionary path
    # The default installation path for mecab-ko-dic by konlpy's script is usually /usr/local/lib/mecab/dic/mecab-ko-dic
    mecab_dic_path = os.path.join(os.sep, 'usr', 'local', 'lib', 'mecab', 'dic', 'mecab-ko-dic')

    # Initialize Mecab. If the dictionary path is valid, use it. Otherwise, try without it.
    # konlpy's Mecab class should now be able to find Tagger from the installed mecab-python3.
    if os.path.exists(mecab_dic_path):
        mecab = Mecab(dicpath=mecab_dic_path)
    else:
        print(f"Warning: MeCab dictionary not found at {mecab_dic_path}. Attempting to initialize Mecab without explicit dicpath.")
        mecab = Mecab()

    kor_corpus = []
    eng_corpus = []

    for kor, eng in cleaned_set:
        kor_clean = preprocessing(kor, "kor")
        eng_clean = preprocessing(eng, "eng")

        # 영문 문장엔 <start>와 <end> 토큰 추가
        eng_tokenized = f"<start> {eng_clean} <end>".split()
        # 한글 토큰화 (KoNLPy Mecab 사용)
        kor_tokenized = mecab.morphs(kor_clean)

        # 토큰 길이 40 이하인 데이터만 선별
        if len(kor_tokenized) <= 40 and len(eng_tokenized) <= 40:
            if len(kor_tokenized) > 0 and len(eng_tokenized) > 2:
                kor_corpus.append(kor_tokenized)
                eng_corpus.append(eng_tokenized)

    return kor_corpus, eng_corpus

# ==========================================
# Helper functions for pre-trained embeddings
# ==========================================

def load_fasttext_embedding_model(model_name, vector_size=300):
    print(f"Downloading/Loading FastText model: {model_name} (this might take a while)...")
    try:
        model = api.load(model_name)
        print(f"Successfully loaded {model_name}.")
        return model
    except ValueError as e:
        print(f"Error loading FastText model {model_name}: {e}")
        print("Please check model_name or ensure sufficient memory/disk space.")
        return None

def create_embedding_matrix(word_index, embedding_model, embedding_dim):
    embedding_matrix = np.zeros((len(word_index) + 1, embedding_dim))
    for word, i in word_index.items():
        try:
            embedding_vector = embedding_model[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            # Words not found in embedding model will be zero-initialized.
            # You could also randomly initialize them if preferred.
            pass
    return embedding_matrix


# Korean-English News Corpus 다운로드 및 로드
repo_dir = 'korean-parallel-corpora'
if not os.path.exists(repo_dir):
    print(f"Cloning repository: https://github.com/jungyeul/{repo_dir}.git")
    !git clone https://github.com/jungyeul/{repo_dir}.git
else:
    print(f"Repository '{repo_dir}' already exists. Skipping clone.")

# Define the data subdirectory and archive name
data_subdir = os.path.join(repo_dir, 'korean-english-news-v1')
archive_name = 'korean-english-park.train.tar.gz'
extracted_kor_file = 'korean-english-park.train.ko'
extracted_eng_file = 'korean-english-park.train.en'

# Check if the extracted files already exist before attempting extraction
if not os.path.exists(os.path.join(data_subdir, extracted_kor_file)):
    print(f"Extracting {os.path.join(data_subdir, archive_name)}")
    # The -C option extracts files into the specified directory
    !tar -xzf {os.path.join(data_subdir, archive_name)} -C {data_subdir}
else:
    print(f"Extracted files '{extracted_kor_file}' and '{extracted_eng_file}' already exist. Skipping extraction.")


# --- Debugging: List contents of korean-parallel-corpora/korean-english-news-v1 to confirm correct paths ---
print(f"\n--- Listing contents of {data_subdir} ---")
!ls -R {data_subdir}
print("---------------------------------------------------\n")
# --- End Debugging ---

# 실제 데이터셋 로드
kor_file_path = os.path.join(data_subdir, extracted_kor_file)
eng_file_path = os.path.join(data_subdir, extracted_eng_file)

with open(kor_file_path, 'r', encoding='utf-8') as f:
    kor_list = f.read().splitlines()

with open(eng_file_path, 'r', encoding='utf-8') as f:
    eng_list = f.read().splitlines()

print(f"원본 한국어 문장 수: {len(kor_list)}")
print(f"원본 영어 문장 수: {len(eng_list)}")

# clean_and_filter_corpus 함수를 사용하여 코퍼스 정제 및 필터링
kor_corpus, eng_corpus = clean_and_filter_corpus(kor_list, eng_list)

print(f"\n정제 및 필터링 후 한국어 문장 수: {len(kor_corpus)}")
print(f"정제 및 필터링 후 영어 문장 수: {len(eng_corpus)}")

print("\n--- 정제된 한국어 코퍼스 샘플 ---")
for i in range(min(5, len(kor_corpus))):
    print(kor_corpus[i])

print("\n--- 정제된 영어 코퍼스 샘플 ---")
for i in range(min(5, len(eng_corpus))):
    print(eng_corpus[i])

# Clean up the cloned repository to save space
# Commenting out for now, as data is needed for subsequent steps.
# !rm -rf {repo_dir}

Repository 'korean-parallel-corpora' already exists. Skipping clone.
Extracted files 'korean-english-park.train.ko' and 'korean-english-park.train.en' already exist. Skipping extraction.

--- Listing contents of korean-parallel-corpora/korean-english-news-v1 ---
korean-parallel-corpora/korean-english-news-v1:
korean-english-park.dev.tar.gz	 korean-english-park.train.ko
korean-english-park.test.tar.gz  korean-english-park.train.tar.gz
korean-english-park.train.en	 README.md
---------------------------------------------------

원본 한국어 문장 수: 94123
원본 영어 문장 수: 94123

정제 및 필터링 후 한국어 문장 수: 61887
정제 및 필터링 후 영어 문장 수: 61887

--- 정제된 한국어 코퍼스 샘플 ---
['온몸', '으로', '말', '하', '는', '기상', '캐스터', '온몸', '으로', '말', '하', '는', '기상', '캐스터']
['야후', '가', '7', '일', '현지', '시간', '총액', '446', '억', '달러', '를', '제시', '한', '마이크', '로', '소프트', 'ms', '의', '인수', '제안', '을', '거부', '했', '다', '.']
['교도', '통신', '은', '고등학생', '인', '이', '소년', '이', '머리', '를', '가방', '에', '넣', '어', '왔', '다고', '보도', '했', '다', '.']
['이번', '영화', '는', '칸

In [5]:
# ==========================================
# Step 3. 데이터 토큰화 (Tokenization & Tensor Conversion)
# ==========================================

def tokenize(corpus, vocab_size=10000):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=vocab_size, filters='')
    tokenizer.fit_on_texts(corpus)

    tensor = tokenizer.texts_to_sequences(corpus)
    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')

    return tensor, tokenizer

# The actual tokenization for the full corpus will now happen in cell 'dd6cdfc1'
# to ensure correct variable scope and order of operations.

### Step 4.1. 모델 초기화 (Model Initialization)

이제 `kor_tensor`와 `eng_tensor`를 통해 얻은 실제 최대 시퀀스 길이와 토크나이저를 기반으로 `encoder`와 `decoder` 모델을 초기화합니다. 이전 셀에서 사용된 더미 데이터의 토크나이저 크기가 아닌, 전체 데이터셋의 토크나이저 크기를 사용해야 합니다.

In [12]:
import tensorflow as tf

class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, embedding_matrix=None):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        if embedding_matrix is not None:
            self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=True)
        else:
            self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')

    def call(self, x, hidden=None):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)
        return output, state

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, embedding_matrix=None):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        if embedding_matrix is not None:
            self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=True)
        else:
            self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.dec_units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state = self.gru(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state, attention_weights


In [13]:
# ==========================================
# Step 4.1. 모델 초기화 (Model Initialization)
# ==========================================

# First, tokenize the actual corpus from cell 12534666
# This ensures that VOCAB_SIZE_KOR_MODEL and VOCAB_SIZE_ENG_MODEL are based on the full dataset.

# Sanity check: Ensure kor_corpus and eng_corpus are populated with the full dataset
# The full corpus (after cleaning) should have 61887 sentences.
if not kor_corpus or len(kor_corpus) < 1000 or not eng_corpus or len(eng_corpus) < 1000:
    print("------------------------------------------------------------------------------------------------------------------")
    print("WARNING: 'kor_corpus' or 'eng_corpus' appear to be empty or too small.")
    print(f"  Current kor_corpus length: {len(kor_corpus) if kor_corpus else 0}")
    print(f"  Current eng_corpus length: {len(eng_corpus) if eng_corpus else 0}")
    print("  Please ensure 'cell 12534666' (Data Loading and Cleaning) has been run successfully to load the full dataset.")
    print("  Then re-run this cell ('dd6cdfc1').")
    print("  The model will now be initialized with incorrect (small) vocabulary and max lengths, which will lead to training issues (e.g., NaN loss).")
    print("------------------------------------------------------------------------------------------------------------------")

VOCAB_SIZE_KOR_DEFAULT = 10000
VOCAB_SIZE_ENG_DEFAULT = 15000

print(f"Tokenizing Korean corpus (length: {len(kor_corpus)}) with vocab_size={VOCAB_SIZE_KOR_DEFAULT}...")
kor_tensor, kor_tokenizer = tokenize(kor_corpus, vocab_size=VOCAB_SIZE_KOR_DEFAULT)
print(f"Tokenizing English corpus (length: {len(eng_corpus)}) with vocab_size={VOCAB_SIZE_ENG_DEFAULT}...")
eng_tensor, eng_tokenizer = tokenize(eng_corpus, vocab_size=VOCAB_SIZE_ENG_DEFAULT)

# Define actual vocabulary sizes from the tokenizers created from the full corpus
VOCAB_SIZE_KOR_MODEL = len(kor_tokenizer.word_index) + 1
VOCAB_SIZE_ENG_MODEL = len(eng_tokenizer.word_index) + 1

# Define actual max lengths from the padded tensors
MAX_LENGTH_KOR = kor_tensor.shape[1]
MAX_LENGTH_ENG = eng_tensor.shape[1]

EMBEDDING_DIM = 300 # FastText common embedding dimension
HIDDEN_DIM = 512 # Consistent with previous use

# Load FastText pre-trained embedding models
print("\nLoading pre-trained FastText embeddings...")
fasttext_kor_model = load_fasttext_embedding_model('cc-ko-300') # Korean FastText model
fasttext_eng_model = load_fasttext_embedding_model('fasttext-wiki-news-subwords-300') # English FastText model

embedding_matrix_kor = None
if fasttext_kor_model:
    embedding_matrix_kor = create_embedding_matrix(kor_tokenizer.word_index, fasttext_kor_model, EMBEDDING_DIM)
    print(f"Korean embedding matrix created with shape: {embedding_matrix_kor.shape}")
else:
    print("WARNING: Korean FastText model 'cc-ko-300' could not be loaded via gensim.downloader.")
    print("         Korean embeddings will be randomly initialized. This may affect initial training stability.")

embedding_matrix_eng = None
if fasttext_eng_model:
    embedding_matrix_eng = create_embedding_matrix(eng_tokenizer.word_index, fasttext_eng_model, EMBEDDING_DIM)
    print(f"English embedding matrix created with shape: {embedding_matrix_eng.shape}")
else:
    print("WARNING: English FastText model 'fasttext-wiki-news-subwords-300' could not be loaded via gensim.downloader.")
    print("         English embeddings will be randomly initialized. This may affect initial training stability.")

# Instantiate Encoder and Decoder models with correct vocabulary sizes and pre-trained embeddings
encoder = Encoder(VOCAB_SIZE_KOR_MODEL, EMBEDDING_DIM, HIDDEN_DIM, embedding_matrix=embedding_matrix_kor)
decoder = Decoder(VOCAB_SIZE_ENG_MODEL, EMBEDDING_DIM, HIDDEN_DIM, embedding_matrix=embedding_matrix_eng)

# Dummy call to build encoder and decoder to ensure their weights are created
# This step requires the tokenizer to have '<start>' in its word_index,
# which should be guaranteed if eng_corpus contains '<start>' tokens and is correctly tokenized.
try:
    dummy_input_enc = tf.zeros((1, MAX_LENGTH_KOR))
    dummy_enc_out, dummy_enc_hidden = encoder(dummy_input_enc)

    dummy_input_dec = tf.expand_dims([eng_tokenizer.word_index['<start>']], 0)
    dummy_dec_out, dummy_dec_hidden, _ = decoder(dummy_input_dec, dummy_enc_hidden, dummy_enc_out)
    print("Models built with dummy input.")
except KeyError as e:
    print(f"ERROR: Could not find '{e}' in English tokenizer. This likely means eng_corpus is not correctly loaded or tokenized.")
except Exception as e:
    print(f"ERROR during model build with dummy input: {e}")

print(f"Encoder initialized with vocab size: {VOCAB_SIZE_KOR_MODEL}, max length: {MAX_LENGTH_KOR}")
print(f"Decoder initialized with vocab size: {VOCAB_SIZE_ENG_MODEL}, max length: {MAX_LENGTH_ENG}")

# Initialize the optimizer AFTER the models are built, with a potentially lower learning rate for stability
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)


Tokenizing Korean corpus (length: 61887) with vocab_size=10000...
Tokenizing English corpus (length: 61887) with vocab_size=15000...

Loading pre-trained FastText embeddings...
Downloading/Loading FastText model: cc-ko-300 (this might take a while)...
Error loading FastText model cc-ko-300: Incorrect model/corpus name
Please check model_name or ensure sufficient memory/disk space.
Downloading/Loading FastText model: fasttext-wiki-news-subwords-300 (this might take a while)...
Successfully loaded fasttext-wiki-news-subwords-300.
         Korean embeddings will be randomly initialized. This may affect initial training stability.
English embedding matrix created with shape: (46117, 300)
Models built with dummy input.
Encoder initialized with vocab size: 45559, max length: 40
Decoder initialized with vocab size: 46117, max length: 40


In [18]:
# ==========================================
# Step 4.2. 모델 훈련 (Training the Model)
# ==========================================

BUFFER_SIZE = len(kor_tensor) # Use the full dataset size for buffer
BATCH_SIZE = 128 # You can adjust this batch size
EPOCHS = 10 # You can adjust the number of epochs for training

# Create a tf.data.Dataset for training
dataset = tf.data.Dataset.from_tensor_slices((kor_tensor, eng_tensor)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

# The encoder and decoder models are now correctly initialized in cell dd6cdfc1
# No need for safeguard initialization here.

print(f"Starting training for {EPOCHS} epochs with batch size {BATCH_SIZE}...")

for epoch in range(EPOCHS):
    total_loss = tf.constant(0.0) # Initialize total_loss as a TensorFlow scalar float
    # Iterate over the dataset, taking batches for training
    for (batch, (inp, targ)) in enumerate(dataset.take(len(kor_tensor)//BATCH_SIZE)):
        # Ensure initial_hidden state for the encoder is passed correctly
        # For GRU, the initial state is typically zeros for the first pass
        batch_loss = train_step(inp, targ, tf.zeros((BATCH_SIZE, HIDDEN_DIM)), encoder, decoder, optimizer, eng_tokenizer)
        total_loss += batch_loss # total_loss will remain a tf.Tensor

        if batch % 100 == 0:
            print(f'Epoch {epoch+1} Batch {batch} Loss {batch_loss.numpy():.4f}')

    print(f'Epoch {epoch+1} Total Loss {(total_loss / (len(kor_tensor)//BATCH_SIZE)).numpy():.4f}') # Convert to numpy for printing

print("Training complete.")

Starting training for 10 epochs with batch size 128...
Epoch 1 Batch 0 Loss 5.9986
Epoch 1 Batch 100 Loss 3.4357
Epoch 1 Batch 200 Loss 3.5423
Epoch 1 Batch 300 Loss 3.5945
Epoch 1 Batch 400 Loss 3.6374
Epoch 1 Total Loss 3.7368
Epoch 2 Batch 0 Loss 3.4055
Epoch 2 Batch 100 Loss 3.2721
Epoch 2 Batch 200 Loss 3.2312
Epoch 2 Batch 300 Loss 3.3457
Epoch 2 Batch 400 Loss 3.3378
Epoch 2 Total Loss 3.3708
Epoch 3 Batch 0 Loss 3.1267
Epoch 3 Batch 100 Loss 3.1404
Epoch 3 Batch 200 Loss 3.0884
Epoch 3 Batch 300 Loss 3.2631
Epoch 3 Batch 400 Loss 3.1176
Epoch 3 Total Loss 3.2694
Epoch 4 Batch 0 Loss 3.4053
Epoch 4 Batch 100 Loss 3.2777
Epoch 4 Batch 200 Loss 3.1450
Epoch 4 Batch 300 Loss 2.9880
Epoch 4 Batch 400 Loss 3.2221
Epoch 4 Total Loss 3.1891
Epoch 5 Batch 0 Loss 3.3988
Epoch 5 Batch 100 Loss 3.1681
Epoch 5 Batch 200 Loss 3.0590
Epoch 5 Batch 300 Loss 3.0719
Epoch 5 Batch 400 Loss 2.8321
Epoch 5 Total Loss 3.1115
Epoch 6 Batch 0 Loss 3.0968
Epoch 6 Batch 100 Loss 3.0142
Epoch 6 Batch 200

## 모델 평가 및 번역 결과 확인 (Model Evaluation and Translation Results)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def evaluate_with_attention(sentence, encoder, decoder, kor_tokenizer, eng_tokenizer, max_length_kor, max_length_eng):
    sentence = preprocessing(sentence, "kor")
    inputs = Mecab().morphs(sentence)
    inputs = kor_tokenizer.texts_to_sequences([inputs])
    inputs = tf.keras.preprocessing.sequence.pad_sequences(inputs, maxlen=max_length_kor, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''
    hidden = tf.zeros((1, HIDDEN_DIM))
    enc_out, enc_hidden = encoder(inputs, hidden)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([eng_tokenizer.word_index['<start>']], 0)

    attention_plot = np.zeros((max_length_eng, inputs.shape[1]))

    for t in range(max_length_eng):
        predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_out)

        attention_weights = tf.reshape(attention_weights, (-1, ))
        attention_plot[t] = attention_weights.numpy()

        predicted_id = tf.argmax(predictions[0]).numpy()

        if eng_tokenizer.index_word.get(predicted_id) == '<end>':
            break

        result += eng_tokenizer.index_word.get(predicted_id, '') + ' '
        dec_input = tf.expand_dims([predicted_id], 0)

    return result.strip(), sentence, attention_plot

def plot_attention_map(attention, sentence, predicted_sentence):
    fig = plt.figure(figsize=(10,10))
    ax = fig.add_subplot(1, 1, 1)
    ax.matshow(attention, cmap='viridis')

    fontdict = {'fontsize': 14}

    ax.set_xticklabels([''] + sentence, fontdict=fontdict, rotation=90)
    ax.set_yticklabels([''] + predicted_sentence, fontdict=fontdict)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show()

In [ ]:
# Evaluate and translate some test sentences
# test_sentences is already in the kernel state based on the context.
print("\n--- Test Translations ---")
for i, sentence in enumerate(test_sentences):
    translated_text, original_sentence_tokens, attention_plot = evaluate_with_attention(
        sentence, encoder, decoder, kor_tokenizer, eng_tokenizer, MAX_LENGTH_KOR, MAX_LENGTH_ENG)
    print(f"Original (Korean): {sentence}")
    print(f"Translated (English): {translated_text}\n")

    # Plot attention map for the first sentence
    if i == 0:
        # Ensure attention_plot is truncated to the length of the predicted sentence tokens
        predicted_tokens = translated_text.split()
        attention_plot = attention_plot[:len(predicted_tokens), :len(original_sentence_tokens)]

        print("\n--- Attention Map for the first sentence ---")
        plot_attention_map(attention_plot, original_sentence_tokens, predicted_tokens)


In [19]:
# ==========================================
# Step 4. 모델 설계 (Attention-based Seq2seq)
# (Class definitions moved to cell 21eb5f28 to ensure latest versions are used during evaluation)
# ==========================================

In [16]:
# ==========================================
# Step 5. 훈련하기 및 평가 (Training & Evaluation)
# ==========================================

# Optimizer will be initialized in a later cell (dd6cdfc1) after models are built
# optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

@tf.function
def train_step(inp, targ, enc_hidden, encoder, decoder, optimizer, eng_tokenizer):
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp)
        dec_hidden = enc_hidden

        # Start token for the batch (<start>)
        dec_input = tf.expand_dims([eng_tokenizer.word_index['<start>']] * inp.shape[0], 1)

        # Teacher forcing - passing the target as the next input
        for t in range(1, targ.shape[1]):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)

            # Update dec_input to target token at time step t
            dec_input = tf.expand_dims(targ[:, t], 1)

    batch_loss = (loss / int(targ.shape[1]))
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

def evaluate_and_translate(sentence, encoder, decoder, kor_tokenizer, eng_tokenizer, max_length_kor, max_length_eng):
    sentence = preprocessing(sentence, "kor")
    inputs = Mecab().morphs(sentence)
    inputs = kor_tokenizer.texts_to_sequences([inputs])
    inputs = tf.keras.preprocessing.sequence.pad_sequences(inputs, maxlen=max_length_kor, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''

    enc_out, enc_hidden = encoder(inputs)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([eng_tokenizer.word_index['<start>']], 0)

    for t in range(max_length_eng):
        predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_out) # Corrected: enc_output to enc_out

        predicted_id = tf.argmax(predictions[0]).numpy()

        if eng_tokenizer.word_index.get(predicted_id, '<end>') == '<end>':
            result += '<end> '
            break

        result += eng_tokenizer.index_word.get(predicted_id, '') + ' '
        dec_input = tf.expand_dims([predicted_id], 0)

    return result.strip()